# Suite PTH — Paths

`Paths.of(store)` names every object a store's roots reach: a schema of a store of schemas by its name, a singleton by
its global name, and any other object by the first route from them, each step `/adjacency[key]`, its key the entry's
values that a unique constraint declares with the object's own link, else its position. A path survives changes that
do not touch the route to the object.

In [ ]:
import { Paths, Proxies, Reflection, Schemas as S, Stores } from "@mbse/schemas/Framework";
import { LookupError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, text } from "./support.js";

const Listed = new S.OfRelation.Builder().name("Listed").links("directory", "contact").create();
const Phones = new S.OfRelation.Builder().name("Phones").links("owner", "phone").properties(text("label")).unique(
  "owner", "label").unique("phone").create(); // the second does not hold the owner
const Slots = new S.OfRelation.Builder().name("Slots").links("owner", "phone").properties(text("slot", BigInt), text("on", Boolean),
  text("amount", Number)).unique("owner", "amount").unique("owner", "slot", "on").create(); // an amount is not written
const Enrolled = new S.OfRelation.Builder().name("Enrolled").links("student", "course", "term").unique("student").unique(
  "student", "term").create(); // neither holds a property
const Directory = new S.OfObject.Builder().name("Directory").ref().singleton("book.Directory").relations(
  (r) => r.name("contacts").of(Listed).me("directory")).create();
const Contact = new S.OfObject.Builder().name("Contact").ref().properties(text("name")).relations(
  (r) => r.name("directories").of(Listed).me("contact"), (r) => r.name("phones").of(Phones).me("owner"),
  (r) => r.name("slots").of(Slots).me("owner"), (r) => r.name("enrolled").of(Enrolled).me("student")).create();
const Phone = new S.OfObject.Builder().name("Phone").ref().properties(text("number")).create();
const Course = new S.OfObject.Builder().name("Course").ref().properties(text("title")).create();
const Archive = new S.OfObject.Builder().name("Archive").ref().singleton("an.Archive").relations(
  (r) => r.name("contacts").of(Listed).me("directory")).create();

/** A store of the schemas, its directory listing Ann, with two phones by label, a slot and an enrollment. */
function book(): [any, any] {
  const store = new Proxies.OfStore() as any;
  for (const schema of [Directory, Archive, Contact, Phone, Course, Listed, Phones, Slots, Enrolled]) store.register(schema);
  const [one, two, three, four] = ["1", "2", "3", "4"].map((n) => store.Phone().number(n).create());
  const [math, fall] = [store.Course().title("math").create(), store.Course().title("fall").create()];
  const ann = store.Contact().name("ann").phones((e: any) => e.phone(one).label("home")).phones(
    (e: any) => e.phone(two).label("work")).slots((e: any) => e.phone(three).slot(7n).on(true).amount(0.5)).slots((e: any) => e.phone(four).slot(7n).on(false)).enrolled(
    (e: any) => e.course(math).term(fall)).create();
  store.Directory(store.singleton("book.Directory")).contacts((e: any) => e.contact(ann)).update();
  return [store, ann];
}

## PTH-01 · Singletons by their global names, other objects by route: keys by unique properties, else by position

In [ ]:
{
  const [store, ann] = book();
  const paths = Paths.of(store);
  assert(paths.of(store.singleton("book.Directory")) === "book.Directory" && paths.of(ann) === "book.Directory/contacts[0]");
  const phones = new Map(store.extent("Phone").map((p: any) => [p.number, paths.of(p)]));
  assert(equal(phones, map({ "1": 'book.Directory/contacts[0]/phones[label="home"]', "2": 'book.Directory/contacts[0]/phones[label="work"]',
    "3": "book.Directory/contacts[0]/slots[on=true,slot=7]", "4": "book.Directory/contacts[0]/slots[on=false,slot=7]" }))); // the first constraint a path writes
  const courses = new Map(store.extent("Course").map((c: any) => [c.title, paths.of(c)]));
  assert(equal(courses, map({ math: "book.Directory/contacts[0]/enrolled[0].course", fall: "book.Directory/contacts[0]/enrolled[0].term" })));
  assert(paths.of(store.singleton("an.Archive")) === "an.Archive" && paths.find("book.Directory/contacts[0]") === ann);
  assert(new Paths.Paths(store).of(ann) === paths.of(ann));
  raises(LookupError, () => paths.find("nowhere"), "no object at 'nowhere'");
  raises(LookupError, () => paths.of(store.Contact().name("loose").create()), "no root reaches the object");
}

## PTH-02 · A path survives changes elsewhere: a property set, an entry added after; a positional one before shifts

In [ ]:
{
  const [store, ann] = book();
  const before = Paths.of(store);
  const home = before.find('book.Directory/contacts[0]/phones[label="home"]');
  store.Phone(home).number("9").update();
  store.Contact(ann).phones((e: any) => e.phone(store.Phone().number("4").create()).label("cell")).update();
  const after = Paths.of(store);
  assert(after.of(home) === before.of(home) && after.of(ann) === before.of(ann));
  const bob = store.Contact().name("bob").create();
  store.Directory(store.singleton("book.Directory")).contacts((e: any) => e.contact(bob)).update();
  assert(Paths.of(store).of(bob) === "book.Directory/contacts[1]" && Paths.of(store).of(ann) === "book.Directory/contacts[0]");
  store.Archive(store.singleton("an.Archive")).contacts((e: any) => e.contact(bob)).update();
  assert(Paths.of(store).of(bob) === "an.Archive/contacts[0]"); // reached first from the root first in name order
}

## PTH-03 · A store of schemas names its schemas; combined stores name the schemas, then the singletons

In [ ]:
{
  const [store, ann] = book();
  const schemas = Reflection.of(store);
  const named = Paths.of(schemas);
  assert(named.of(Contact as never) === "Contact" && named.find("Phones") === (Phones as never) && named.of(Directory as never) === "Directory");
  const both = new Stores.Combined(schemas, store);
  const combined = Paths.of(both);
  assert(combined.of(Contact as never) === "Contact" && combined.of(ann) === "book.Directory/contacts[0]");
  assert(Paths.of(new Stores.Combined(new Stores.Combined(schemas), store)).of(Phone as never) === "Phone"); // combined again
}